# RAUQ Attention Example

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    from transformers import AutoTokenizer
except ImportError:
    AutoTokenizer = None

In [ ]:
RUN_PATH = Path(os.environ.get('RAUQ_RUN_PATH', ''))
TAD_STATS_NAME = 'UAD_max_meanlog_max'
MANAGER_FILE = 'ue_manager_seed1'
MODEL_NAME = 'meta-llama/Llama-3.1-8B'

example_index_env = os.environ.get('EXAMPLE_INDEX')
EXAMPLE_INDEX = int(example_index_env) if example_index_env not in (None, '') else None  # None picks example 814 when available, otherwise the last example
LAYER = 30
UNCERTAINTY_THRESHOLD = 0.7
OUTPUT_PATH = Path(os.environ.get('RAUQ_OUTPUT_PATH', 'attention_first_example.png'))

if not str(RUN_PATH):
    raise ValueError('Set RUN_PATH or RAUQ_RUN_PATH before running the notebook.')

manager_path = RUN_PATH / MANAGER_FILE
tad_stats_path = RUN_PATH / 'tad_stats' / TAD_STATS_NAME

In [ ]:
def get_metric_values(manager, preferred=('AlignScore', 'AlignScoreInv', 'Comet', 'Accuracy')):
    gen_metrics = manager.get('gen_metrics', {})
    for metric in preferred:
        key = ('sequence', metric)
        if key in gen_metrics:
            return metric, np.asarray(gen_metrics[key])
    raise KeyError(f'None of {preferred} found in manager["gen_metrics"].')


def load_attention(example_index):
    path = tad_stats_path / f'attentions_{example_index}.npy'
    if not path.exists():
        raise FileNotFoundError(path)
    return np.load(path)


def token_labels(text, model_name=MODEL_NAME):
    if AutoTokenizer is None:
        return text.split()
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, local_files_only=True)
        token_ids = tokenizer(text)['input_ids']
        return tokenizer.batch_decode(token_ids, skip_special_tokens=False)
    except Exception:
        return text.split()


def question_from_input(text):
    if 'Q:' in text and 'A:' in text:
        return text.split('Q:')[-1].split('A:')[0].strip()
    if 'Text:' in text and 'Summary' in text:
        return text.split('Text:')[-1].split('Summary')[0].strip()[:220]
    return text[:220]

def choose_index(requested, total):
    if total <= 0:
        raise ValueError('No examples found in manager metrics.')
    if requested is None:
        return min(814, total - 1)
    if requested < 0 or requested >= total:
        raise IndexError(f'EXAMPLE_INDEX={requested} outside [0, {total - 1}].')
    return requested


def choose_layer(requested, attention):
    attention = np.asarray(attention)
    if attention.ndim == 4:
        n_layers = attention.shape[1]
    elif attention.ndim == 3:
        n_layers = attention.shape[0] if attention.shape[0] <= 128 else attention.shape[1]
    else:
        n_layers = 1
    return min(requested, n_layers - 1)


def attention_tokens_by_heads(attention, layer):
    attention = np.asarray(attention)
    if attention.ndim == 4:
        # Saved by RAUQ as (batch/examples, layers, heads, tokens).
        return attention[0, layer].T
    if attention.ndim == 3:
        # Either (layers, heads, tokens) or already (tokens, layers, heads).
        if attention.shape[0] <= 128:
            return attention[layer].T
        return attention[:, layer, :]
    if attention.ndim == 2:
        return attention
    raise ValueError(f'Unsupported attention shape: {attention.shape}')


In [ ]:
data = torch.load(manager_path, weights_only=False)
metric_name, quality = get_metric_values(data)
example_index = choose_index(EXAMPLE_INDEX, len(quality))
attention = load_attention(example_index)
layer = choose_layer(LAYER, attention)

x = attention_tokens_by_heads(attention, layer)
selected_head = x.mean(axis=0).argmax()
uncertainty = 1 - x[:, selected_head]

answer = data['stats']['greedy_texts'][example_index].strip()
target = data['stats']['target_texts'][example_index]
question = question_from_input(data['stats']['input_texts'][example_index])

print(f'Model answer: {answer!r}')
print(f'Target answer: {target}')
print(f'{metric_name}: {quality[example_index]:.4f}')
print(f'Selected head: layer={layer}, head={selected_head}')

In [ ]:
def plot_attention_example(x, uncertainty, question, answer, threshold=UNCERTAINTY_THRESHOLD):
    labels = token_labels(answer)
    if len(labels) == x.shape[0] + 1:
        labels = labels[1:]
    labels = labels[:x.shape[0]]
    if len(labels) < x.shape[0]:
        labels += [f'token_{i}' for i in range(len(labels), x.shape[0])]

    fig, ax = plt.subplots(figsize=(20, 10))
    image = ax.imshow(x, cmap='coolwarm', aspect='auto')
    im_ratio = x.shape[0] / max(x.shape[1], 1)
    fig.colorbar(image, ax=ax, fraction=0.047 * im_ratio, pad=0.04)

    ax.set_xlabel('Attention Head', fontsize=26)
    ax.set_ylabel('Tokens in Answer', fontsize=26)
    ax.set_title(f'Question: {question}', fontsize=18)
    ax.set_yticks(range(x.shape[0]))
    ax.set_yticklabels(labels, fontsize=14)
    ax.tick_params(axis='x', labelsize=14)

    for value, label in zip(uncertainty, ax.get_yticklabels()):
        label.set_color('red' if value > threshold else 'green')

    return fig, ax

fig, ax = plot_attention_example(x, uncertainty, question, answer)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, bbox_inches='tight', dpi=200)
print(f'Saved figure to {OUTPUT_PATH}')
plt.show()